In [1]:
!git clone https://github.com/ShreeNithiB/Amazon--ML---challenge-

Cloning into 'Amazon--ML---challenge-'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 7.13 KiB | 7.13 MiB/s, done.


In [2]:
%cd Amazon--ML---challenge-

/content/Amazon--ML---challenge-


In [3]:
!ls

amazon-ml-challenge-dataset-training.ipynb


In [5]:
# ============================================================
# STEP 4.5: RE-INITIALIZE NOTEBOOK IMPORTS
# ============================================================

from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
import gc
import time
from collections import defaultdict

DATA_DIR = Path("/content/")
OUTPUT_DIR = Path("/kaggle/working/amazon_ml_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")
print("DATA_DIR:", DATA_DIR)


Environment initialized successfully.
DATA_DIR: /content


In [7]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
import gc
import time
from collections import defaultdict

# Assuming DATA_DIR is already defined from a previous cell, if not, it should be defined here as well.
# DATA_DIR = Path("/content/")

LEGAL_SUFFIX_MAP = {
    "incorporated": "inc",
    "corporation": "corp",
    "company": "co",
    "limited": "ltd",
    "private": "pvt",
    "llc": "llc",
    "llp": "llp",
}

def normalize_unicode(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    return text


def normalize_name(text):
    text = normalize_unicode(text)

    text = re.sub(
        r"[^\\w\\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    tokens = [
        LEGAL_SUFFIX_MAP.get(token, token)
        for token in tokens
    ]

    return " ".join(tokens)


def build_name_country_index(file_path, chunk_size=250_000):

    index = defaultdict(list)

    start = time.time()
    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\\t",
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=chunk_size,
        encoding='latin1'
    ):

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        for row in chunk.itertuples(index=False):

            if not row.name_norm:
                continue

            key = (
                row.name_norm,
                row.country
            )

            index[key].append(
                row.entity_id
            )

        total_rows += len(chunk)

        print(
            f"Processed {total_rows:,} rows",
            end="\\r"
        )

        del chunk
        gc.collect()

    elapsed = (
        time.time() - start
    ) / 60

    print()

    print(
        f"Processed: {total_rows:,} rows"
    )

    print(
        f"Unique name+country keys: "
        f"{len(index):,}"
    )

    print(
        f"Time: {elapsed:.2f} minutes"
    )

    return index

# Rebuilding Source 3 index with correct DATA_DIR
print("Rebuilding Source 3 index...")
s3_index = build_name_country_index(
    DATA_DIR / "train_source3.tsv"
)

Rebuilding Source 3 index...


/tmp/ipykernel_34223/68143875.py:67: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  for chunk in pd.read_csv(


Processed 250,000 rows\rProcessed 500,000 rows\rProcessed 750,000 rows\rProcessed 1,000,000 rows\rProcessed 1,250,000 rows\rProcessed 1,500,000 rows\rProcessed 1,750,000 rows\rProcessed 2,000,000 rows\rProcessed 2,250,000 rows\rProcessed 2,500,000 rows\rProcessed 2,750,000 rows\rProcessed 3,000,000 rows\rProcessed 3,250,000 rows\rProcessed 3,500,000 rows\rProcessed 3,750,000 rows\rProcessed 4,000,000 rows\rProcessed 4,250,000 rows\rProcessed 4,500,000 rows\rProcessed 4,750,000 rows\rProcessed 5,000,000 rows\rProcessed 5,250,000 rows\rProcessed 5,285,603 rows\r
Processed: 5,285,603 rows
Unique name+country keys: 2,781
Time: 1.53 minutes


In [8]:
# ============================================================
# STEP 2: NORMALIZATION FUNCTIONS
# ============================================================

LEGAL_SUFFIX_MAP = {
    "incorporated": "inc",
    "corporation": "corp",
    "company": "co",
    "limited": "ltd",
    "private": "pvt",
    "llc": "llc",
    "llp": "llp",
}

def normalize_unicode(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    return text


def normalize_name(text):
    text = normalize_unicode(text)

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    tokens = [
        LEGAL_SUFFIX_MAP.get(token, token)
        for token in tokens
    ]

    return " ".join(tokens)


def normalize_address(text):
    text = normalize_unicode(text)

    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\blane\b": "ln",
        r"\bdrive\b": "dr",
        r"\bhighway\b": "hwy",
        r"\bparkway\b": "pkwy",
        r"\bboulevard\b": "blvd",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(
            pattern,
            replacement,
            text
        )

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


print("Normalization functions ready!")
print(normalize_name("ABC Corporation Pvt. Ltd."))
print(normalize_address("500 Market Street, San Jose"))

Normalization functions ready!
abc corp pvt ltd
500 market st san jose


In [10]:
# ============================================================
# STEP 3: INDEX BUILDER
# ============================================================

def build_name_country_index(file_path, chunk_size=250_000):

    index = defaultdict(list)

    start = time.time()
    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=chunk_size,
        encoding='latin1'  # Added encoding parameter
    ):

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        for row in chunk.itertuples(index=False):

            if not row.name_norm:
                continue

            key = (
                row.name_norm,
                row.country
            )

            index[key].append(
                row.entity_id
            )

        total_rows += len(chunk)

        print(
            f"Processed {total_rows:,} rows",
            end="\r"
        )

        del chunk
        gc.collect()

    elapsed = (
        time.time() - start
    ) / 60

    print()

    print(
        f"Processed: {total_rows:,} rows"
    )

    print(
        f"Unique name+country keys: "
        f"{len(index):,}"
    )

    print(
        f"Time: {elapsed:.2f} minutes"
    )

    return index


print("Index builder ready!")

Index builder ready!


In [12]:
# ============================================================
# STEP 5: BUILD SOURCE 3 INDEX
# ============================================================

print("Building Source 3 index...")

s3_index = build_name_country_index(
    DATA_DIR / "train_source3.tsv"
)

Building Source 3 index...
Processed 5,285,603 rows
Processed: 5,285,603 rows
Unique name+country keys: 4,223,574
Time: 1.78 minutes


In [9]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
import gc
import time
from collections import defaultdict

DATA_DIR = Path("/content/")

LEGAL_SUFFIX_MAP = {
    "incorporated": "inc",
    "corporation": "corp",
    "company": "co",
    "limited": "ltd",
    "private": "pvt",
    "llc": "llc",
    "llp": "llp",
}

def normalize_unicode(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    return text


def normalize_name(text):
    text = normalize_unicode(text)

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    tokens = [
        LEGAL_SUFFIX_MAP.get(token, token)
        for token in tokens
    ]

    return " ".join(tokens)


def build_name_country_index(file_path, chunk_size=250_000):

    index = defaultdict(list)

    start = time.time()
    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=chunk_size,
        encoding='latin1'
    ):

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        for row in chunk.itertuples(index=False):

            if not row.name_norm:
                continue

            key = (
                row.name_norm,
                row.country
            )

            index[key].append(
                row.entity_id
            )

        total_rows += len(chunk)

        print(
            f"Processed {total_rows:,} rows",
            end="\r"
        )

        del chunk
        gc.collect()

    elapsed = (
        time.time() - start
    ) / 60

    print()

    print(
        f"Processed: {total_rows:,} rows"
    )

    print(
        f"Unique name+country keys: "
        f"{len(index):,}"
    )

    print(
        f"Time: {elapsed:.2f} minutes"
    )

    return index




Building Source 2 index...
Processed 5,045,413 rows
Processed: 5,045,413 rows
Unique name+country keys: 3,916,623
Time: 1.68 minutes


In [11]:
# ============================================================
# STEP 4: BUILD SOURCE 2 INDEX
# ============================================================

print("Building Source 2 index...")

s2_index = build_name_country_index(
    DATA_DIR / "train_source2.tsv"
)

Building Source 2 index...
Processed 5,045,413 rows
Processed: 5,045,413 rows
Unique name+country keys: 3,916,623
Time: 2.02 minutes


In [12]:
# ============================================================
# STEP 5: BUILD SOURCE 3 INDEX
# ============================================================

print("Building Source 3 index...")

s3_index = build_name_country_index(
    DATA_DIR / "train_source3.tsv"
)

Building Source 3 index...
Processed 5,285,603 rows
Processed: 5,285,603 rows
Unique name+country keys: 4,187,382
Time: 2.06 minutes


In [15]:
# ============================================================
# STEP 6: VERIFY
# ============================================================

print("s2_index exists:", "s2_index" in globals())
print("s3_index exists:", "s3_index" in globals())
print("normalize_name exists:", "normalize_name" in globals())

print(
    "S2 index keys:",
    len(s2_index)
)

print(
    "S3 index keys:",
    len(s3_index)
)

s2_index exists: True
s3_index exists: True
normalize_name exists: True
S2 index keys: 3916623
S3 index keys: 4187382


In [16]:
# ============================================================
# STEP 7: LOAD 10,000 SOURCE 1 RECORDS
# ============================================================

S1_SAMPLE_N = 10_000

s1_sample = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    nrows=S1_SAMPLE_N,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s1_sample["name_norm"] = (
    s1_sample["business_name"]
    .map(normalize_name)
)

print(
    f"Loaded Source 1 records: "
    f"{len(s1_sample):,}"
)

display(s1_sample.head())

Loaded Source 1 records: 10,000


,entity_id,business_name,business_address,country,name_norm
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center


In [17]:
# ============================================================
# STEP 8: GENERATE CANDIDATES
# Blocking rule:
#     normalized business name + country
# ============================================================

candidate_rows = []

start_time = time.time()

for row in s1_sample.itertuples(index=False):

    key = (row.name_norm, row.country)

    # Source 2 candidates
    for candidate_id in s2_index.get(key, []):
        candidate_rows.append({
            "source1_entity_id": row.entity_id,
            "candidate_entity_id": candidate_id,
            "candidate_source": "S2"
        })

    # Source 3 candidates
    for candidate_id in s3_index.get(key, []):
        candidate_rows.append({
            "source1_entity_id": row.entity_id,
            "candidate_entity_id": candidate_id,
            "candidate_source": "S3"
        })

elapsed = time.time() - start_time

candidates = pd.DataFrame(
    candidate_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "candidate_source"
    ]
)

print(f"Candidate generation time: {elapsed:.2f} seconds")
print(f"Total candidate pairs: {len(candidates):,}")

print("\nCandidates by source:")
print(candidates["candidate_source"].value_counts())

print("\nAverage candidates per Source 1:")
print(len(candidates) / len(s1_sample))

Candidate generation time: 0.27 seconds
Total candidate pairs: 109,408

Candidates by source:
candidate_source
S3    58203
S2    51205
Name: count, dtype: int64

Average candidates per Source 1:
10.9408


In [18]:
# ============================================================
# STEP 9: LOAD GROUND TRUTH
# ============================================================

GT_PATH = DATA_DIR / "train_ground_truth.tsv"

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    usecols=[
        "source1_entity_id",
        "matched_entity_ids"
    ]
)

gt_sample = gt[
    gt["source1_entity_id"].isin(
        s1_sample["entity_id"]
    )
].copy()

print(f"Ground-truth Source 1 entities: {len(gt_sample):,}")

display(gt_sample.head())

Ground-truth Source 1 entities: 1,829


,source1_entity_id,matched_entity_ids
128,S1-424976769,"S2-18126063,S3-443109567,S3-296472077"
382,S1-874660306,"S2-279857860,S3-289365741"
748,S1-539879230,"S2-83796020,S3-945547250,S3-127246506,S3-11765..."
795,S1-840168502,NaN
827,S1-716830359,"S2-795817041,S2-306163710,S3-762649005,S3-7375..."


In [19]:
# ============================================================
# STEP 10: EXPLODE GROUND TRUTH INTO MATCH PAIRS
# ============================================================

gt_pairs = []

for row in gt_sample.itertuples(index=False):

    if pd.isna(row.matched_entity_ids):
        continue

    matched_ids = str(row.matched_entity_ids).split(",")

    for matched_id in matched_ids:

        matched_id = matched_id.strip()

        if matched_id:
            gt_pairs.append({
                "source1_entity_id": row.source1_entity_id,
                "matched_entity_id": matched_id,
                "source": matched_id[:2]
            })

gt_pairs = pd.DataFrame(
    gt_pairs,
    columns=[
        "source1_entity_id",
        "matched_entity_id",
        "source"
    ]
)

print(f"Total true match pairs: {len(gt_pairs):,}")

print("\nTrue matches by source:")
print(gt_pairs["source"].value_counts())

display(gt_pairs.head(10))

Total true match pairs: 6,283

True matches by source:
source
S3    3259
S2    3024
Name: count, dtype: int64


,source1_entity_id,matched_entity_id,source
0,S1-424976769,S2-18126063,S2
1,S1-424976769,S3-443109567,S3
2,S1-424976769,S3-296472077,S3
3,S1-874660306,S2-279857860,S2
4,S1-874660306,S3-289365741,S3
5,S1-539879230,S2-83796020,S2
6,S1-539879230,S3-945547250,S3
7,S1-539879230,S3-127246506,S3
8,S1-539879230,S3-117651018,S3
9,S1-716830359,S2-795817041,S2


In [20]:
# ============================================================
# STEP 11: CALCULATE CANDIDATE RECALL
# ============================================================

candidate_pairs = candidates.rename(
    columns={
        "candidate_entity_id": "matched_entity_id"
    }
)[
    [
        "source1_entity_id",
        "matched_entity_id"
    ]
].drop_duplicates()

# Match candidates against ground truth
recall_check = gt_pairs.merge(
    candidate_pairs,
    on=[
        "source1_entity_id",
        "matched_entity_id"
    ],
    how="left",
    indicator=True
)

recall_check["found"] = (
    recall_check["_merge"] == "both"
)

total_true = len(recall_check)
retrieved_true = recall_check["found"].sum()
missed_true = total_true - retrieved_true

overall_recall = (
    retrieved_true / total_true
    if total_true > 0
    else 0
)

print("=" * 60)
print("CANDIDATE RECALL")
print("=" * 60)

print(f"Total true match pairs : {total_true:,}")
print(f"Retrieved true matches : {retrieved_true:,}")
print(f"Missed true matches    : {missed_true:,}")
print(f"Candidate recall       : {overall_recall:.4%}")

CANDIDATE RECALL
Total true match pairs : 6,283
Retrieved true matches : 1,585
Missed true matches    : 4,698
Candidate recall       : 25.2268%


In [21]:
# ============================================================
# STEP 12: SOURCE-SPECIFIC RECALL
# ============================================================

for source in ["S2", "S3"]:

    subset = recall_check[
        recall_check["source"] == source
    ]

    total = len(subset)
    found = subset["found"].sum()

    recall = (
        found / total
        if total > 0
        else 0
    )

    print(
        f"{source} candidate recall: {recall:.4%}"
    )
    print(
        f"{source} true pairs: {total:,}"
    )
    print(
        f"{source} retrieved: {found:,}"
    )
    print()

S2 candidate recall: 24.9669%
S2 true pairs: 3,024
S2 retrieved: 755

S3 candidate recall: 25.4679%
S3 true pairs: 3,259
S3 retrieved: 830



In [22]:
# ============================================================
# STEP 13: CANDIDATE COUNT DISTRIBUTION
# ============================================================

candidate_counts = (
    candidates
    .groupby(
        ["source1_entity_id", "candidate_source"]
    )
    .size()
    .unstack(fill_value=0)
)

if "S2" not in candidate_counts.columns:
    candidate_counts["S2"] = 0

if "S3" not in candidate_counts.columns:
    candidate_counts["S3"] = 0

candidate_counts["TOTAL"] = (
    candidate_counts["S2"] +
    candidate_counts["S3"]
)

print("Candidate statistics:")
print(candidate_counts["TOTAL"].describe())

print("\nCandidate count percentiles:")

display(
    candidate_counts["TOTAL"].quantile(
        [
            0.00,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00
        ]
    )
)

Candidate statistics:
count    7417.000000
mean       14.750977
std        34.693696
min         1.000000
25%         1.000000
50%         2.000000
75%        12.000000
max       458.000000
Name: TOTAL, dtype: float64

Candidate count percentiles:


,TOTAL
0.00,1.00
0.25,1.00
0.50,2.00
0.75,12.00
0.90,34.00
0.95,75.00
0.99,184.84
1.00,458.00


In [28]:
# ============================================================
# STEP 14A: INSTALL FAISS
# ============================================================

!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.1 MB/s eta 0:00:00


In [23]:
# ============================================================
# STEP 14B: VERIFY FAISS
# ============================================================

import faiss

print("FAISS installed successfully!")
print("FAISS version:", faiss.__version__)

FAISS installed successfully!
FAISS version: 1.15.1


In [24]:
# ============================================================
# STEP 15: CHECK KERNEL STATE
# ============================================================

print("s2_index exists:", "s2_index" in globals())
print("s3_index exists:", "s3_index" in globals())
print("normalize_name exists:", "normalize_name" in globals())
print("s1_sample exists:", "s1_sample" in globals())

import faiss
print("FAISS version:", faiss.__version__)

s2_index exists: True
s3_index exists: True
normalize_name exists: True
s1_sample exists: True
FAISS version: 1.15.1


In [25]:
# ============================================================
# STEP 16A: PREPARE UNIQUE NAMES FOR FUZZY RETRIEVAL
# ============================================================

import numpy as np
import pandas as pd
import time
import gc

print("Preparing unique normalized names...")

# Get unique keys from our existing exact indexes
s2_unique_names = list(s2_index.keys())
s3_unique_names = list(s3_index.keys())

# Each key is:
# (normalized_name, country)

s2_name_country = pd.DataFrame(
    s2_unique_names,
    columns=["name_norm", "country"]
)

s3_name_country = pd.DataFrame(
    s3_unique_names,
    columns=["name_norm", "country"]
)

print(f"S2 unique name+country keys: {len(s2_name_country):,}")
print(f"S3 unique name+country keys: {len(s3_name_country):,}")

display(s2_name_country.head())

Preparing unique normalized names...
S2 unique name+country keys: 3,916,623
S3 unique name+country keys: 4,187,382


,name_norm,country
0,à à 3 4à à à 3 4à à à à à à à à à aà à à 3 4à ...,India
1,holloway peak inc seafood,US
2,à à à à à à à aà à à à aà à à à à à à 2à à 2à aà,India
3,summit inc,US
4,delta tetlecommunication inc,US


In [26]:
# ============================================================
# STEP 17: BUILD COUNTRY-SPECIFIC TF-IDF INDEXES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import gc
import time

def build_country_tfidf_indexes(name_country_df):
    """
    Build sparse character TF-IDF + cosine nearest-neighbor
    indexes separately for each country.
    """

    indexes = {}

    countries = (
        name_country_df["country"]
        .dropna()
        .unique()
    )

    print(f"Countries found: {list(countries)}")

    for country in countries:

        print(f"\nBuilding index for: {country}")

        subset = name_country_df[
            name_country_df["country"] == country
        ].copy()

        subset = subset.reset_index(drop=True)

        print(
            f"Unique names: {len(subset):,}"
        )

        # Check if there are enough documents for TF-IDF Vectorizer (min_df is 2)
        if len(subset) < 2:
            print(f"Skipping TF-IDF index for {country} due to insufficient unique names ({len(subset):,})")
            continue # Skip to the next country

        vectorizer = TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=2,
            dtype=np.float32
        )

        start = time.time()

        matrix = vectorizer.fit_transform(
            subset["name_norm"]
        )

        print(
            f"TF-IDF shape: {matrix.shape}"
        )

        nn = NearestNeighbors(
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        )

        nn.fit(matrix)

        elapsed = time.time() - start

        indexes[country] = {
            "names": subset,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "nn": nn
        }

        print(
            f"Finished {country} "
            f"in {elapsed / 60:.2f} minutes"
        )

        del subset
        gc.collect()

    return indexes


In [27]:
# ============================================================
# FIX: RECREATE SOURCE 2 NAME + COUNTRY TABLE
# ============================================================

print("Recreating s2_name_country from existing s2_index...")

s2_unique_names = list(s2_index.keys())

s2_name_country = pd.DataFrame(
    s2_unique_names,
    columns=["name_norm", "country"]
)

print(
    f"S2 unique name+country keys: "
    f"{len(s2_name_country):,}"
)

display(s2_name_country.head())

Recreating s2_name_country from existing s2_index...
S2 unique name+country keys: 3,916,623


,name_norm,country
0,à à 3 4à à à 3 4à à à à à à à à à aà à à 3 4à ...,India
1,holloway peak inc seafood,US
2,à à à à à à à aà à à à aà à à à à à à 2à à 2à aà,India
3,summit inc,US
4,delta tetlecommunication inc,US


In [28]:
# ============================================================
# FIX: REBUILD SOURCE 3 EXACT INDEX
# ============================================================

print("Rebuilding Source 3 exact index...")

s3_index = build_name_country_index(
    DATA_DIR / "train_source3.tsv"
)

print("\nSource 3 exact index rebuilt!")

Rebuilding Source 3 exact index...
Processed 5,285,603 rows
Processed: 5,285,603 rows
Unique name+country keys: 4,187,382
Time: 2.43 minutes

Source 3 exact index rebuilt!


In [29]:
# ============================================================
# RECREATE SOURCE 3 NAME + COUNTRY TABLE
# ============================================================

s3_unique_names = list(s3_index.keys())

s3_name_country = pd.DataFrame(
    s3_unique_names,
    columns=["name_norm", "country"]
)

print(
    f"S3 unique name+country keys: "
    f"{len(s3_name_country):,}"
)

display(s3_name_country.head())

S3 unique name+country keys: 4,187,382


,name_norm,country
0,wilfordhancock com,US
1,international south consultants pvt ltd,India
2,llc moncada lã arning center,US
3,moyna s coffee,US
4,pvt efs print ventures ltd,India


In [30]:
# ============================================================
# STEP 19: BUILD SOURCE 3 FUZZY INDEX
# ============================================================

print("Building Source 3 fuzzy indexes...")

s3_fuzzy_indexes = build_country_tfidf_indexes(
    s3_name_country
)

print("\nSource 3 fuzzy indexes ready!")

Building Source 3 fuzzy indexes...
Countries found: ['US', 'India']

Building index for: US
Unique names: 2,563,766
TF-IDF shape: (2563766, 1001759)
Finished US in 3.87 minutes

Building index for: India
Unique names: 1,623,616
TF-IDF shape: (1623616, 661722)
Finished India in 2.64 minutes

Source 3 fuzzy indexes ready!


In [31]:
# ============================================================
# STEP 14A: CHECK SOURCE 2 STATE
# ============================================================

print("s2_index exists:", "s2_index" in globals())
print("s2_name_country exists:", "s2_name_country" in globals())

s2_index exists: True
s2_name_country exists: True


In [32]:
# ============================================================
# STEP 14B: RECREATE SOURCE 2 NAME + COUNTRY TABLE
# ============================================================

print("Recreating s2_name_country...")

s2_unique_names = list(s2_index.keys())

s2_name_country = pd.DataFrame(
    s2_unique_names,
    columns=["name_norm", "country"]
)

print(
    f"S2 unique name+country keys: "
    f"{len(s2_name_country):,}"
)

display(s2_name_country.head())

Recreating s2_name_country...
S2 unique name+country keys: 3,916,623


,name_norm,country
0,à à 3 4à à à 3 4à à à à à à à à à aà à à 3 4à ...,India
1,holloway peak inc seafood,US
2,à à à à à à à aà à à à aà à à à à à à 2à à 2à aà,India
3,summit inc,US
4,delta tetlecommunication inc,US


In [33]:
# ============================================================
# STEP 14C: RECREATE FUZZY INDEX FUNCTION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import time
import gc
import numpy as np
import pandas as pd


def build_country_tfidf_indexes(name_country_df):
    """
    Build sparse character TF-IDF + cosine nearest-neighbor
    indexes separately for each country.
    """

    indexes = {}

    countries = (
        name_country_df["country"]
        .dropna()
        .unique()
    )

    print(f"Countries found: {list(countries)}")

    for country in countries:

        print(f"\nBuilding index for: {country}")

        subset = name_country_df[
            name_country_df["country"] == country
        ].copy()

        subset = subset.reset_index(drop=True)

        print(
            f"Unique names: {len(subset):,}"
        )

        vectorizer = TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=2,
            dtype=np.float32
        )

        start = time.time()

        matrix = vectorizer.fit_transform(
            subset["name_norm"]
        )

        print(
            f"TF-IDF shape: {matrix.shape}"
        )

        nn = NearestNeighbors(
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        )

        nn.fit(matrix)

        elapsed = time.time() - start

        indexes[country] = {
            "names": subset,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "nn": nn
        }

        print(
            f"Finished {country} "
            f"in {elapsed / 60:.2f} minutes"
        )

        del subset
        gc.collect()

    return indexes


print("build_country_tfidf_indexes() is ready!")

build_country_tfidf_indexes() is ready!


In [34]:
# ============================================================
# STEP 14D: CHECK SOURCE 2 DATA
# ============================================================

print("s2_index exists:", "s2_index" in globals())
print("s2_name_country exists:", "s2_name_country" in globals())
print("s3_fuzzy_indexes exists:", "s3_fuzzy_indexes" in globals())

if "s2_name_country" in globals():
    print(
        "S2 unique name+country keys:",
        len(s2_name_country)
    )

s2_index exists: True
s2_name_country exists: True
s3_fuzzy_indexes exists: True
S2 unique name+country keys: 3916623


### Steps to Push your Colab Notebook to GitHub

1.  **Save your current Colab notebook**: Make sure your current notebook is saved (e.g., `File` -> `Save` or `File` -> `Save a copy in GitHub`). For the following steps, let's assume you save it to the `/content/` directory with the name `my_amazon_ml_challenge.ipynb`.

2.  **Set up a local Git repository**: We'll clone your target repository into a new directory and then copy your notebook into it.

In [9]:
# IMPORTANT: Before running, ensure your notebook is saved (File -> Save)!

# 1. Change directory into the repository where the notebook resides
%cd /content/Amazon--ML---challenge-/

# 2. Configure your Git user name and email (if you haven't already)
# Replace with your actual name and email
!git config user.email "your_email@example.com"
!git config user.name "Your Name"

# 3. Add the notebook file, commit, and push
# Replace <YOUR_PERSONAL_ACCESS_TOKEN> with your actual GitHub Personal Access Token
# If your default branch is 'master', change 'main' to 'master' in the push command
!git add "amazon-ml-challenge-dataset-training.ipynb" # Explicitly add the notebook file
!git commit -m "Update Colab notebook: amazon-ml-challenge-dataset-training.ipynb"
!git push "https://<YOUR_PERSONAL_ACCESS_TOKEN>@github.com/ShreeNithiB/Amazon--ML---challenge-.git" main

print("--------------------------------------------------------------------------------")
print("Please ensure you've saved your notebook (File -> Save) before running.")
print("Then, replace <YOUR_PERSONAL_ACCESS_TOKEN> with your actual GitHub Personal Access Token.")
print("Also, replace 'your_email@example.com' and 'Your Name' with your Git credentials.")
print("If your default branch is 'master', change 'main' to 'master' in the git push command.")
print("Re-run this cell after making these changes.")

/content/Amazon--ML---challenge-
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://%3CYOUR_PERSONAL_ACCESS_TOKEN%3E@github.com': No such device or address
--------------------------------------------------------------------------------
Please ensure you've saved your notebook (File -> Save) before running.
Then, replace <YOUR_PERSONAL_ACCESS_TOKEN> with your actual GitHub Personal Access Token.
Also, replace 'your_email@example.com' and 'Your Name' with your Git credentials.
If your default branch is 'master', change 'main' to 'master' in the git push command.
Re-run this cell after making these changes.


### Important Notes:

*   **GitHub Credentials**: When you run `!git push`, you will likely be prompted to enter your GitHub username and Personal Access Token (PAT). GitHub no longer supports password authentication for Git operations. You'll need to generate a PAT from your GitHub settings if you don't have one.
*   **Notebook Filename**: Ensure you replace `my_amazon_ml_challenge.ipynb` with the exact name you saved your Colab notebook as.
*   **Branch Name**: The command `!git push origin main` assumes your repository's default branch is `main`. If it's `master` (or another name), adjust the command accordingly.

In [6]:
!git push


fatal: could not read Username for 'https://github.com': No such device or address


In [7]:
!git status

On branch main
Your branch is based on 'origin/main', but the upstream is gone.
  (use "git branch --unset-upstream" to fixup)

nothing to commit, working tree clean
